In [ ]:
!pip install --quiet accelerate
!pip install --quiet bitsandbytes
!pip install --quiet datasets
!pip install --quiet -U nnsight
!pip install --quiet -U pyvis
!pip install --quiet -U ray
!pip install --quiet llm-fleet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
value = os.environ.pop('CUDA_VISIBLE_DEVICES', None)

In [ ]:
import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from huggingface_hub import hf_hub_download, login

from nnsight import NNsight, LanguageModel, util

import torch
from torch import Tensor, nn

import numpy as np

import math
import json
import random
import zlib
import base64
import ast
import tokenize
import itertools
import io
import re
import gc
import pickle

import pandas as pd

from abc import ABC, abstractmethod
from dataclasses import dataclass, field, asdict
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from tqdm import tqdm
import json

import threading
from queue import Empty, Queue

In [ ]:
from fleet import FleetWorker, VectorDSU, Node, ResidualCollection
from fleet.prior_tree import AgglomerativePriorTreeBuilder

In [ ]:
MAX_ATTEMPTS = 32
SAMPLING = "fleet" # or "temperature"
"""
ground_truth - rewarded and evaluated by ground truth verifier
orm - rewarded and evaluated by orm
mixed - rewarded by orm, evaluated by ground truth verifier
diversity-sampling - no reward signals, evaluated by ground truth verifier
"""

STRATEGY = "orm"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
HYPERPARAMETERS = {
    'layer': 26,
    'temperature': 3.1,
    'dsu': VectorDSU(threshold=0.9),
    'threshold': (0.11, 0.05),
    'prior_tree': None
}

In [ ]:
device_count = torch.cuda.device_count()
devices = []

for i in range(device_count):
    devices.append(f"cuda:{i}")

In [ ]:
devices

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
hf_token = ''

In [ ]:
login(token=hf_token)

In [ ]:
from datasets import load_dataset

lcb_latest_filenames = [
    "test.jsonl",
    "test2.jsonl",
    "test3.jsonl",
    "test4.jsonl",
    "test5.jsonl",
    "test6.jsonl",
]

repo_id = "livecodebench/code_generation_lite"
lcb_latest_files = [
    hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset", force_download=False) for filename in lcb_latest_filenames
]

In [ ]:
lcb_codegen = []
for file in lcb_latest_files:
    with open(file, "r") as file:
        for idx, line in enumerate(file):
            line_data = json.loads(line)
            lcb_codegen.append(line_data)

In [ ]:
print(len(lcb_codegen))

In [ ]:
lcb_codegen[0]

In [ ]:
from datetime import datetime

cutoff = datetime(2023, 12, 1)
lcb_coding_tasks_prefilter = [item for item in lcb_codegen if datetime.strptime(item['contest_date'], "%Y-%m-%dT%H:%M:%S") > cutoff]

question_ids = set()
lcb_coding_tasks = []
for task in lcb_coding_tasks_prefilter:
    if task['question_title'] in question_ids:
      continue

    if task['difficulty'] != 'easy':
        continue

    task['task_id'] = task['question_title']
    task['source'] = "livecodebench"
    task['metadata'] = ast.literal_eval(task['metadata'])
    
    question_ids.add(task['task_id'])
    lcb_coding_tasks.append(task)

print(len(lcb_coding_tasks))

In [ ]:
repo_id = "openai/gsm8k"
gsm_dataset = load_dataset(repo_id, 'main', split='test')
gsm_few_shots = load_dataset(repo_id, 'main', split='train')

In [ ]:
math_tasks = [item for item in gsm_dataset]

def clean_answer(question):
    return re.sub(r"<<.*>>", "", question)

def get_few_shot_prompt(examples_count=2):
    random_examples = []

    for _ in range(examples_count):
        example_id = random.randint(1, len(gsm_few_shots)) - 1
        random_examples.append(example_id)
    
    few_shot_items = gsm_few_shots.select(random_examples)

    few_shot_pieces = []
    for f in few_shot_items:
        few_shot_prompt = f"Question: {f['question']}\nAnswer: {clean_answer(f['answer'])}\n\n"
        few_shot_pieces.append(few_shot_prompt)

    few_shot_prompt = "".join(few_shot_pieces)

    return few_shot_prompt

for i, item in enumerate(math_tasks):
    item['task_id'] = f'gsm8k_{i}'
    item['source'] = 'gsm8k'
    item['prompt'] = get_few_shot_prompt() + f"Question: {item['question']}\nAnswer:"

print(math_tasks[0])

In [ ]:
%%writefile eval_dto.py
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional
import numpy as np
from scipy.special import comb

class Stats(BaseModel):
    all_stats: List[Dict]  = Field(default_factory=list)
    best_stats: Dict = Field(default_factory=lambda: {'correct': False, 'score': 0.0, 'reward': 0.0, 'completion': ""})
    
    def add(self, stats):
        self.all_stats.append(stats)
        if stats['reward'] > self.best_stats['reward']:
            self.best_stats = stats

    # As ORM is not GT verifier, reward going up does not always translate into higher accuracy
    # Such variant allows to see whether this result in model deceiving the ORM or finiding consistent strategy 
    def pass_k_orm(self, k):
        current_stats = self.all_stats[:k]
        current_best = max(current_stats, key=lambda s: s['reward'])

        return 1.0 if current_best['correct'] else 0.0

    def pass_k_noisy(self, k):
        c = len([s for s in self.all_stats[:k] if s['correct']])

        return 1.0 if c > 0 else 0.0

    def rm_k(self, k):
        rm_scores = np.array(s['reward'] for s in self.all_stats)
        gt_labels = np.array([s['correct'] for s in self.all_stats])
        N = len(self.all_stats)

        sort_indices = np.argsort(rm_scores)[::-1]
        sorted_gt = gt_labels[sort_indices]
            
        j = np.arange(N)
        worse_items = N - 1 - j
        results = {}
        
        ways_to_be_best = comb(worse_items, k - 1, exact=False)
        total_combinations = comb(N, k, exact=False)
        expected_pass = np.sum(sorted_gt * ways_to_be_best) / total_combinations

        return expected_pass
        
    def pass_k(self, k):
        n = len(self.all_stats)
        c = len([s for s in self.all_stats if s['correct']])
    
        if n - c < k:
            return 1.0
        
        return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

class TaskState(BaseModel):
    task_id: str
    task_data: Any
    conversation: List[Dict[str, str]] = Field(default_factory=list)
    agent_states: Optional[List[Any]] = None
    attempts: int = 0
    is_done: bool = False
    stats: Stats = Field(default_factory=dict)

class BenchmarkResults(BaseModel):
    items: List[TaskState]
    metadata: Dict[str, Any]

In [ ]:
from eval_dto import *

In [ ]:
os.environ['FLASHINFER_DISABLE_VERSION_CHECK'] = '1'

In [ ]:
from fleet import FleetWorker, VectorDSU, Node, ResidualCollection
from fleet.prior_tree import AgglomerativePriorTreeBuilder
from typing import List, Optional, Tuple, Dict, Any, Union
import copy

In [ ]:
import ray
ray.init(num_gpus=2, log_to_driver=True, ignore_reinit_error=True)

In [ ]:
%%writefile lcb_test_suite.py
# @title
# https://github.com/LiveCodeBench/LiveCodeBench/blob/998c52d394b836f15fff3b9a29866191108ff81b/lcb_runner/evaluation/testing_util.py
#

import ast
import json
import sys
import faulthandler
import platform

# used for debugging to time steps
from datetime import datetime

# to run the solution files we're using a timing based approach
import signal

from io import StringIO

# used for testing the code that reads from input
from unittest.mock import patch, mock_open

# from pyext import RuntimeModule
from types import ModuleType

from enum import Enum
from decimal import Decimal
import time

import multiprocessing
import traceback

import_string = "from string import *\nfrom re import *\nfrom datetime import *\nfrom collections import *\nfrom heapq import *\nfrom bisect import *\nfrom copy import *\nfrom math import *\nfrom random import *\nfrom statistics import *\nfrom itertools import *\nfrom functools import *\nfrom operator import *\nfrom io import *\nfrom sys import *\nfrom json import *\nfrom builtins import *\nfrom typing import *\nimport string\nimport re\nimport datetime\nimport collections\nimport heapq\nimport bisect\nimport copy\nimport math\nimport random\nimport statistics\nimport itertools\nimport functools\nimport operator\nimport io\nimport sys\nimport json\nsys.setrecursionlimit(50000)\n"


def truncatefn(s, length=300):
    if isinstance(s, str):
        pass
    else:
        s = str(s)
    if len(s) <= length:
        return s

    return s[: length // 2] + "...(truncated) ..." + s[-length // 2 :]


class CODE_TYPE(Enum):
    call_based = 0
    standard_input = 1


# stuff for setting up signal timer
class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    print("timeout occured: alarm went off")
    raise TimeoutException


# used to capture stdout as a list
# from https://stackoverflow.com/a/16571630/6416660
# alternative use redirect_stdout() from contextlib
class Capturing(list):
    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self._stringio = StringIO()
        # Make closing the StringIO a no-op
        self._stringio.close = lambda x: 1
        return self

    def __exit__(self, *args):
        self.append(self._stringio.getvalue())
        del self._stringio  # free up some memory
        sys.stdout = self._stdout


def clean_if_name(code: str) -> str:
    try:
        astree = ast.parse(code)
        last_block = astree.body[-1]
        if isinstance(last_block, ast.If):
            condition = last_block.test
            if ast.unparse(condition).strip() == "__name__ == '__main__'":
                code = (
                    ast.unparse(astree.body[:-1]) + "\n" + ast.unparse(last_block.body)  # type: ignore
                )
    except:
        pass

    return code


def make_function(code: str) -> str:
    try:
        import_stmts = []
        all_other_stmts = []
        astree = ast.parse(code)
        for stmt in astree.body:
            if isinstance(stmt, (ast.Import, ast.ImportFrom)):
                import_stmts.append(stmt)
            else:
                all_other_stmts.append(stmt)

        function_ast = ast.FunctionDef(
            name="wrapped_function",
            args=ast.arguments(
                posonlyargs=[], args=[], kwonlyargs=[], kw_defaults=[], defaults=[]
            ),
            body=all_other_stmts,
            decorator_list=[],
            lineno=-1,
        )
        main_code = (
            import_string
            + "\n"
            + ast.unparse(import_stmts)  # type: ignore
            + "\n"
            + ast.unparse(function_ast)  # type: ignore
        )
        return main_code
    except Exception as e:
        return code


def call_method(method, inputs):

    if isinstance(inputs, list):
        inputs = "\n".join(inputs)

    inputs_line_iterator = iter(inputs.split("\n"))

    # sys.setrecursionlimit(10000)

    # @patch('builtins.input', side_effect=inputs.split("\n"))
    @patch("builtins.open", mock_open(read_data=inputs))
    @patch("sys.stdin", StringIO(inputs))
    @patch("sys.stdin.readline", lambda *args: next(inputs_line_iterator))
    @patch("sys.stdin.readlines", lambda *args: inputs.split("\n"))
    @patch("sys.stdin.read", lambda *args: inputs)
    # @patch('sys.stdout.write', print)
    def _inner_call_method(_method):
        try:
            return _method()
        except SystemExit as e:
            pass
        finally:
            pass

    return _inner_call_method(method)


def get_function(compiled_sol, fn_name: str):  # type: ignore
    try:
        assert hasattr(compiled_sol, fn_name)
        return getattr(compiled_sol, fn_name)
    except Exception as e:
        return


def compile_code(code: str, timeout: int):
    signal.alarm(timeout)
    try:
        tmp_sol = ModuleType("tmp_sol", "")
        try:
            exec(code, tmp_sol.__dict__)
        except Exception as e:
            print(f"Error executing code: {e}")
            # print(f"     executed:\n{code}")
            raise
        if "class Solution" in code:
            # leetcode wraps solutions in `Solution`
            # this is a hack to check if it is leetcode solution or not
            # currently livecodebench only supports LeetCode but
            # else condition allows future extensibility to other platforms
            compiled_sol = tmp_sol.Solution()
        else:
            # do nothing in the other case since function is accesible
            compiled_sol = tmp_sol

        assert compiled_sol is not None
    finally:
        signal.alarm(0)

    return compiled_sol


def convert_line_to_decimals(line: str) -> tuple[bool, list[Decimal]]:
    try:
        decimal_line = [Decimal(elem) for elem in line.split()]
    except:
        return False, []
    return True, decimal_line


def get_stripped_lines(val: str):
    ## you don't want empty lines to add empty list after splitlines!
    val = val.strip()

    return [val_line.strip() for val_line in val.split("\n")]


def grade_call_based(
    code: str, all_inputs: list, all_outputs: list, fn_name: str, timeout: int
):
    # call-based clean up logic
    # need to wrap in try-catch logic after to catch the correct errors, but for now this is fine.
    code = import_string + "\n\n" + code
    compiled_sol = compile_code(code, timeout)

    if compiled_sol is None:
        return

    method = get_function(compiled_sol, fn_name)

    if method is None:
        return

    all_inputs = [
        [json.loads(line) for line in inputs.split("\n")] for inputs in all_inputs
    ]

    all_outputs = [json.loads(output) for output in all_outputs]

    total_execution = 0
    all_results = []
    for idx, (gt_inp, gt_out) in enumerate(zip(all_inputs, all_outputs)):
        signal.alarm(timeout)
        faulthandler.enable()
        try:
            # can lock here so time is useful
            start = time.time()
            prediction = method(*gt_inp)
            total_execution += time.time() - start
            signal.alarm(0)

            # don't penalize model if it produces tuples instead of lists
            # ground truth sequences are not tuples
            if isinstance(prediction, tuple):
                prediction = list(prediction)

            tmp_result = prediction == gt_out

            # handle floating point comparisons

            all_results.append(tmp_result)

            if not tmp_result:
                return all_results, {
                    "output": truncatefn(prediction),
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                    "error_code": -2,
                    "error_message": "Wrong Answer",
                }
        except Exception as e:
            signal.alarm(0)
            if "timeoutexception" in repr(e).lower():
                all_results.append(-3)
                return all_results, {
                    "error": repr(e),
                    "error_code": -3,
                    "error_message": "Time Limit Exceeded",
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                }
            else:
                all_results.append(-4)
                return all_results, {
                    "error": repr(e),
                    "error_code": -4,
                    "error_message": "Runtime Error",
                    "inputs": truncatefn(gt_inp),
                    "expected": truncatefn(gt_out),
                }

        finally:
            signal.alarm(0)
            faulthandler.disable()

    return all_results, {"execution time": total_execution}


def grade_stdio(
    code: str,
    all_inputs: list,
    all_outputs: list,
    timeout: int,
):
    ## runtime doesn't interact well with __name__ == '__main__'
    code = clean_if_name(code)

    ## we wrap the given code inside another function
    code = make_function(code)

    compiled_sol = compile_code(code, timeout)
    if compiled_sol is None:
        return

    method = get_function(compiled_sol, "wrapped_function")

    if method is None:
        return

    all_results = []
    total_execution_time = 0
    for idx, (gt_inp, gt_out) in enumerate(zip(all_inputs, all_outputs)):
        signal.alarm(timeout)
        faulthandler.enable()

        signal.alarm(timeout)
        with Capturing() as captured_output:
            try:
                start = time.time()
                call_method(method, gt_inp)
                total_execution_time += time.time() - start
                # reset the alarm
                signal.alarm(0)
            except Exception as e:
                signal.alarm(0)
                if "timeoutexception" in repr(e).lower():
                    all_results.append(-3)
                    return all_results, {
                        "error": repr(e),
                        "error_code": -3,
                        "error_message": "Time Limit Exceeded",
                        "inputs": truncatefn(gt_inp),
                        "expected": truncatefn(gt_out),
                    }
                else:
                    all_results.append(-4)
                    return all_results, {
                        "error": repr(e),
                        "error_code": -4,
                        "error_message": "Runtime Error",
                        "inputs": truncatefn(gt_inp),
                        "expected": truncatefn(gt_out),
                    }

            finally:
                signal.alarm(0)
                faulthandler.disable()

        prediction = captured_output[0]

        stripped_prediction_lines = get_stripped_lines(prediction)
        stripped_gt_out_lines = get_stripped_lines(gt_out)

        ## WA happens in multiple circumstances
        ## so cache the return to make it clean!
        WA_send_args = {
            "output": truncatefn(prediction),
            "inputs": truncatefn(gt_inp),
            "expected": truncatefn(gt_out),
            "error_code": -2,
        }

        if len(stripped_prediction_lines) != len(stripped_gt_out_lines):
            all_results.append(-2)
            WA_send_args["error_message"] = "Wrong answer: mismatched output length"
            return all_results, WA_send_args

        for output_line_idx, (
            stripped_prediction_line,
            stripped_gt_out_line,
        ) in enumerate(zip(stripped_prediction_lines, stripped_gt_out_lines)):
            WA_send_args["error_message"] = (
                f"Wrong answer at {output_line_idx=}: {truncatefn(stripped_prediction_line)} != {truncatefn(stripped_gt_out_line)}"
            )

            ## CASE 1: exact match
            if stripped_prediction_line == stripped_gt_out_line:
                continue

            ## CASE 2: element-wise comparision
            ## if there are floating elements
            ## use `decimal` library for good floating point comparision
            ## otherwise gotcha: np.isclose(50000000000000000, 50000000000000001) = True
            ## note that we should always be able to convert to decimals

            success, decimal_prediction_line = convert_line_to_decimals(
                stripped_prediction_line
            )
            if not success:
                all_results.append(-2)
                return all_results, WA_send_args
            success, decimal_gtout_line = convert_line_to_decimals(stripped_gt_out_line)
            if not success:
                all_results.append(-2)
                return all_results, WA_send_args

            if decimal_prediction_line == decimal_gtout_line:
                continue

            all_results.append(-2)
            return all_results, WA_send_args
        all_results.append(True)

    return all_results, {"execution time": total_execution_time}


def run_test(sample, tests, test=None, debug=False, timeout=6):
    """
    if test(generated_code) is not None it'll try to run the code.
    otherwise it'll just return an input and output pair.
    """
    signal.signal(signal.SIGALRM, timeout_handler)

    # Disable functionalities that can make destructive changes to the test.
    # max memory is set to 4GB
    reliability_guard()

    if debug:
        print(f"start = {datetime.now().time()}")

    try:
        in_outs = json.loads(tests)
    except ValueError as e:
        raise e
        in_outs = None

    if in_outs:
        if in_outs.get("fn_name") is None:
            which_type = CODE_TYPE.standard_input  # Standard input
            method_name = None

        else:
            which_type = CODE_TYPE.call_based  # Call-based
            method_name = in_outs["fn_name"]

    if debug:
        print(f"loaded input_output = {datetime.now().time()}")

    if test is None:
        assert False, "should not happen: test code is none"
        return in_outs, {"error": "no test code provided"}
    elif test is not None:
        results = []
        sol = import_string
        if debug:
            print(f"loading test code = {datetime.now().time()}")

        if which_type == CODE_TYPE.call_based:
            signal.alarm(timeout)
            try:
                results, metadata = grade_call_based(
                    code=test,
                    all_inputs=in_outs["inputs"],
                    all_outputs=in_outs["outputs"],
                    fn_name=method_name,
                    timeout=timeout,
                )
                return results, metadata
            except Exception as e:
                return [-4], {
                    "error_code": -4,
                    "error_message": f"Error during testing: {traceback.format_exc()}",
                }
            finally:
                signal.alarm(0)
        elif which_type == CODE_TYPE.standard_input:
            # sol
            # if code has if __name__ == "__main__": then remove it

            signal.alarm(timeout)
            try:
                results, metadata = grade_stdio(
                    code=test,
                    all_inputs=in_outs["inputs"],
                    all_outputs=in_outs["outputs"],
                    timeout=timeout,
                )
                return results, metadata
            except Exception as e:
                return [-4], {
                    "error_code": -4,
                    "error_message": f"Error during testing: {traceback.format_exc()}",
                }
            finally:
                signal.alarm(0)


def reliability_guard(maximum_memory_bytes=None):
    """
    This disables various destructive functions and prevents the generated code
    from interfering with the test (e.g. fork bomb, killing other processes,
    removing filesystem files, etc.)
    WARNING
    This function is NOT a security sandbox. Untrusted code, including, model-
    generated code, should not be blindly executed outside of one. See the
    Codex paper for more information about OpenAI's code sandbox, and proceed
    with caution.
    """

    if maximum_memory_bytes is not None:
        import resource

        resource.setrlimit(
            resource.RLIMIT_AS, (maximum_memory_bytes, maximum_memory_bytes)
        )
        resource.setrlimit(
            resource.RLIMIT_DATA, (maximum_memory_bytes, maximum_memory_bytes)
        )
        if not platform.uname().system == "Darwin":
            resource.setrlimit(
                resource.RLIMIT_STACK, (maximum_memory_bytes, maximum_memory_bytes)
            )

    faulthandler.disable()

    import builtins

    # builtins.exit = None
    builtins.quit = None

    import os

    os.environ["OMP_NUM_THREADS"] = "1"

    os.kill = None
    os.system = None
    os.putenv = None
    os.remove = None
    os.removedirs = None
    os.rmdir = None
    os.fchdir = None
    os.setuid = None
    os.fork = None
    os.forkpty = None
    os.killpg = None
    os.rename = None
    os.renames = None
    os.truncate = None
    os.replace = None
    os.unlink = None
    os.fchmod = None
    os.fchown = None
    os.chmod = None
    os.chown = None
    os.chroot = None
    os.fchdir = None
    os.lchflags = None
    os.lchmod = None
    os.lchown = None
    os.getcwd = None
    os.chdir = None

    import shutil

    shutil.rmtree = None
    shutil.move = None
    shutil.chown = None

    import subprocess

    subprocess.Popen = None  # type: ignore

    builtins.help = None

    import sys

    sys.modules["ipdb"] = None
    sys.modules["joblib"] = None
    sys.modules["resource"] = None
    sys.modules["psutil"] = None
    sys.modules["tkinter"] = None


def _temp_run(sample, generation, tests, debug, result, metadata_list, timeout):
    res, metadata = run_test(sample, tests, test=generation, debug=debug, timeout=timeout)
    result.append(res)
    metadata_list.append(metadata)


def check_correctness(sample, generation, tests, timeout, debug=False):
    """Check correctness of code generation with a global timeout.
    The global timeout is to catch some extreme/rare cases not handled by the timeouts
    inside `run_test`"""

    manager = multiprocessing.Manager()
    result = manager.list()
    metadata_list = manager.list()
    p = multiprocessing.Process(
        target=_temp_run,
        args=(sample, generation, tests, debug, result, metadata_list, timeout),
    )

    p.start()
    p.join(
        timeout=(timeout + 1) * len(json.loads(tests)["inputs"]) + 5
    )

    if p.is_alive():
        p.kill()

    if not result:
        in_outs = json.loads(tests)
        # consider that all tests failed
        result = [[-1 for i in range(len(in_outs["inputs"]))]]
        if debug:
            print("global timeout")

    return result[0], metadata_list[0], result, metadata_list



In [ ]:
%%writefile gsm8k_test_suite.py

import re
import sympy
from sympy.parsing.latex import parse_latex

ANS_RE_GSM8k = re.compile(r"#### (\-?[\$0-9\.\,]+)")
INVALID_ANS_GSM8k = "[invalid]"
GSM8K_IGNORE_REGEXES = [",", "\\$", "\\.$"]

def is_equiv(x1: str, x2: str) -> bool:
    """
    x1 and x2 are normalized latex string
    """
    try:
        with timeout(seconds=5):
            try:
                parsed_x1 = parse_latex(x1)
                parsed_x2 = parse_latex(x2)
            except (
                sympy.parsing.latex.errors.LaTeXParsingError,
                sympy.SympifyError,
                TypeError,
            ):
                return False

            try:
                diff = parsed_x1 - parsed_x2
            except TypeError:
                return False

            try:
                if sympy.simplify(diff) == 0:
                    return True
                else:
                    return False
            except ValueError:
                return False
    except TimeoutError:
        return False
    except ImportError as e:
        return False
    except Exception as e:
        return False

def filter_ignores(st, regexes_to_ignore):
    if regexes_to_ignore is not None:
        for s in regexes_to_ignore:
            st = re.sub(s, "", st)
    return st


def extract_answer_gsm8k(completion):
    match = ANS_RE_GSM8k.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = filter_ignores(
            match_str,
            GSM8K_IGNORE_REGEXES,
        )
        return match_str
    else:
        return INVALID_ANS_GSM8k

def evaluate_gsm8k(task_data, extended_convo, verbose=False):
    in_answer = extract_answer_gsm8k(extended_convo[-1]['content'])
    gt_answer = extract_answer_gsm8k(task_data['answer'])
    return in_answer != INVALID_ANS_GSM8k and (in_answer == gt_answer or is_equiv(in_answer, gt_answer))

In [ ]:
from gsm8k_test_suite import *

In [ ]:
%%writefile fleet_ray_worker.py
from fleet import Node, FleetWorker, VectorDSU, ResidualCollection, Trajectory
from fleet.prior_tree import AgglomerativePriorTreeBuilder
import torch
from torch import Tensor, nn

import numpy as np

import math

from typing import List, Optional, Tuple, Dict, Any, Union

import copy
import traceback

import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig, GenerationConfig
from huggingface_hub import hf_hub_download, login

from nnsight import NNsight, LanguageModel, util

import json
import random
import zlib
import base64
import ast
import tokenize
import itertools
import io
import os
import re
import gc
import pickle
import time

import pandas as pd

from abc import ABC, abstractmethod
from dataclasses import dataclass, field, asdict
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from tqdm import tqdm
import json

import threading
from queue import Empty, Queue

import ray

from lcb_test_suite import *
from gsm8k_test_suite import *
from pydantic import BaseModel, Field, ConfigDict, field_serializer, field_validator
from eval_dto import *

import warnings

def load_test_cases(sample):
  public_test_cases = json.loads(sample["public_test_cases"])  # type: ignore

  if "private_test_cases" in sample:
      try:
          private_test_cases = json.loads(sample["private_test_cases"])  # type: ignore
      except:
          private_test_cases = json.loads(
              pickle.loads(
                  zlib.decompress(
                      base64.b64decode(sample["private_test_cases"].encode("utf-8"))  # type: ignore
                  )
              )
          )  # type: ignore
  else:
      private_test_cases = []
    
  tests = json.dumps(
      {
          "inputs": [
              t["input"]
              for t in public_test_cases + private_test_cases
          ],
          "outputs": [
              t["output"]
              for t in public_test_cases + private_test_cases
          ],
          "fn_name": sample["metadata"].get("func_name", None),
      }
  )

  return tests

def extract_code(text):
    pattern = r".*```python\n(.*?)\n```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches is None or len(matches) == 0:
        return ""
    initial_string = matches[-1] + "\n"

    return initial_string

def evaluate_task(task, conversation, verbose=True):
  final_answer = extract_code(conversation[-1]["content"])
  pred_python_code = final_answer.replace("```python", "").replace("```", "")

  if verbose:
      print(final_answer)
      print(pred_python_code)
    
  if "def " not in pred_python_code:
    if verbose:
      print("No def found in the last code snippet.")
    return {
      "correct": False,
      "pass@1": 0,
      "score": 0,
      "response": "No def found in the last code snippet."
    }
    
  # Adding imports for HE-derived samples
  if "prompt" in task and task.get('source', '') == 'humaneval':
    # Extract imports from sample["prompt"] -- this affects full
    prompt_ast = ast.parse(task["prompt"])
    imports = []
    for node in prompt_ast.body:
      if isinstance(node, (ast.Import, ast.ImportFrom)):
        imports.append(ast.unparse(node))

    # Prepend imports to pred_python_func
    if imports:
      pred_python_code = "\n".join(imports) + "\n\n" + pred_python_code

  # Force update the function name with the true function name
  old_func_name = pred_python_code.split("def ")[1].split("(")[0].strip()
  if task.get('source', '') == 'gsm8k_coding': # There was a split of gsm8k tasks rewritten to ask for a program that will solve the problem described
      pred_python_code = pred_python_code.replace(f"{old_func_name}()", f"{task["metadata"]["func_name"]}(compatibility_arg=None)", 1)
      pred_python_code = pred_python_code.replace(old_func_name, task["metadata"]["func_name"])
  elif "func_name" in task["metadata"]:
      pred_python_code = pred_python_code.replace(old_func_name, task["metadata"]["func_name"])

  # load tests
  testcases = load_test_cases(task)

  output, metadata, full_output, full_metadata = check_correctness(task, pred_python_code, testcases, timeout=6)

  if verbose:
    print(full_output)
    print(full_metadata)

    for i, (output, metadata) in enumerate(zip(full_output, full_metadata)):
      print(f"{i}.")
      print(f"Expected: {metadata.get('expected', '')}")
      print("\n")
      print(f"Actual: {metadata.get('output', '')}")
      print("\n")

  all_test_cases_passed = all(o is True for o in output)

  score = len([o for o in output if o is True]) / len(output)
  return {
    "correct": all_test_cases_passed,
    "pass@1": 1 if all_test_cases_passed else 0,
    "score": score,
    "response": full_metadata
  }
 
def temperature_based_sampler(
    model, tokenizer, dtype, conversations, temperature=1.0, verbose=True, generation_kwargs=None
):
    def map_messages(messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        return messages

    conversations = [map_messages(m) for m in conversations]

    tokenizer.padding_side = "left"
    generation_prompt = tokenizer.apply_chat_template(
        conversations, 
        tokenize=True,
        padding=True,
        truncation=True,
        max_length=1024,
        return_tensors='pt',
        return_dict=True,
        add_generation_prompt=True
    )
    prompt_tokens = generation_prompt['input_ids']
    attention_mask = generation_prompt['attention_mask']

    offset = prompt_tokens.shape[1]
    
    if generation_kwargs is None:
        generation_kwargs = dict(max_new_tokens=1024, do_sample=True, temperature=temperature)

    with model.generate(generation_prompt.to(model.device), **generation_kwargs) as tracer:
        outputs = model.generator.output.save()

    results = []
    for i in range(len(conversations)):
        result = {
            'completion': tokenizer.decode(outputs[i][offset:], skip_special_tokens=True),
        }
        results.append(result)

    return results

def entropy_search_based_sampler(
    model, tokenizer, workers, dtype, conversations, verbose=True, generation_kwargs=None
):
    def map_messages(messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        return messages

    conversations = [map_messages(m) for m in conversations]

    tokenizer.padding_side = "left"
    generation_prompt = tokenizer.apply_chat_template(
        conversations, 
        tokenize=True,
        padding=True,
        truncation=True,
        max_length=1024,
        return_tensors='pt',
        return_dict=True,
        add_generation_prompt=True
    )
    prompt_tokens = generation_prompt['input_ids']
    attention_mask = generation_prompt['attention_mask']

    for i, worker in enumerate(workers):
        worker.update_prompt(prompt_tokens[i][attention_mask[i] == 1].tolist())

    intercept_layer = model.model.layers[worker.layer]

    if generation_kwargs is None:
        generation_kwargs = dict(max_new_tokens=1024, do_sample=False)

    hit_eos = [False for w in workers]
    
    with model.generate(generation_prompt, **generation_kwargs) as tracer:
        with tracer.iter[:] as iterator:
            if iterator > 0:
                input_tokens = model.model.embed_tokens.input
                for i, worker in enumerate(workers):
                    if hit_eos[i]:
                        continue
                    
                    token = input_tokens[i][-1].detach().item().save()
                    worker.update_tokens(token)

                    hit_eos[i] = token == tokenizer.eos_token

            layer_output = intercept_layer.output
            activation = layer_output[:, -1].detach()
            all_logits = model.lm_head(model.model.norm(activation))
            logits = all_logits.detach().save()
            
            hit_threshold = []
            for i, worker in enumerate(workers):
                if hit_eos[i]:
                    continue
                    
                hit_threshold.append(worker.update_logits(activation[i], logits[i].unsqueeze(0)))

            lm_output = model.lm_head.output[:, -1].detach()
            penalties = torch.zeros_like(lm_output)

            for i, worker in enumerate(workers):
                if hit_eos[i]:
                    continue
                
                p_ids = []
                if hit_threshold:
                    penalties, p_ids = worker.process_logits(lm_output[i])
    
                if verbose and len(p_ids) > 0:
                    print(lm_output[p_ids], lm_output.argmax(), tokenizer.decode([lm_output.argmax()]))
                model.lm_head.output[i, -1] = model.lm_head.output[i, -1] - penalties
                if verbose and len(p_ids) > 0:
                    print(
                        model.lm_head.output[i, -1][p_ids], 
                        model.lm_head.output[i, -1].argmax(), 
                        tokenizer.decode([model.lm_head.output[i, -1].argmax()])
                    )

    results = []
    for worker in workers:
        result = {
            'changes': [(act.cpu(), token) for act, token in worker.queue],
            'completion': tokenizer.decode(worker.tokens[worker.offset:], skip_special_tokens=True),
        }
        results.append(result)

    return results

@ray.remote(num_gpus=0.5)
class RewardActor(object):
    def __init__(self):
        model_name = "Skywork/Skywork-Reward-V2-Llama-3.2-3B"
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map="cuda:0",
            num_labels=1,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def get_reward(self, conversation):
        conv_formatted = self.tokenizer.apply_chat_template(conversation, tokenize=False)
        if self.tokenizer.bos_token is not None and conv_formatted.startswith(self.tokenizer.bos_token):
            conv_formatted = conv_formatted[len(self.tokenizer.bos_token):]
        conv_tokenized = self.tokenizer(conv_formatted, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            score = self.model(**conv_tokenized).logits[0][0]
            normalized_score = torch.sigmoid((score - 10) / 5)

        del conv_tokenized
        gc.collect()
        torch.cuda.empty_cache()
        
        return normalized_score.item()

@ray.remote(num_gpus=1)
class FleetActor(object):
    def __init__(self, model_name, dtype):        
        self.model = LanguageModel(model_name, dtype=dtype, device_map="cuda:0", dispatch=True)
        self.dtype = dtype
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

    def fleet_job(self, 
        fleet_task_producer, output_queue, 
        batch_size, max_attempts, verbose, return_trajectory, evaluation_mode, reward_model
    ):
        batch = []
        
        while True:
            while len(batch) < 4:
                data = ray.get(fleet_task_producer.get.remote())
                if data is None:
                    break

                task, context = data
                old_dsu = context['dsu']
                context['dsu'] = context['dsu'].to('cuda:0')
                del old_dsu
                batch.append((task, context))

            if len(batch) == 0:
                break

            tasks, contexts = tuple(map(list, zip(*batch)))
            results = self.fleet_task(tasks, batch_size, max_attempts, contexts, verbose, return_trajectory, evaluation_mode, reward_model)

            pending_tasks, finished_tasks = [], []
            peak_mem, exec_time, tasks, contexts, residual_collections = results
            for task, context, residual_collection in zip(tasks, contexts, residual_collections):
                if task.stats.best_stats['correct'] and evaluation_mode != "orm":
                    # residual_collection.dsu = VectorDSU() # to reduce the footprint
                    residual_collection.dsu = residual_collection.dsu.to('cpu')
                    finished_tasks.append((peak_mem, exec_time, task, residual_collection))
                    continue

                if task.attempts == max_attempts:
                    # residual_collection.dsu = VectorDSU() # to reduce the footprint
                    residual_collection.dsu = residual_collection.dsu.to('cpu')
                    finished_tasks.append((peak_mem, exec_time, task, residual_collection))
                    continue

                pending_tasks.append((task, context))

            for finished_task in finished_tasks:
                output_queue.put(finished_task)
            batch = pending_tasks
    
    def fleet_task(
        self, tasks, batch_size, max_attempts, contexts,
        verbose, return_trajectory, evaluation_mode, reward_model
    ):
        solutions = [[] for t in tasks]
        peak_mem = {i: [] for i in range(max_attempts * batch_size)}
        exec_time = {i: [] for i in range(max_attempts * batch_size)}

        dsus = {t.task_id: context['dsu'] for t, context in zip(tasks, contexts)}
        residual_collections = {t.task_id: ResidualCollection(dsu=dsus[t.task_id]) for t in tasks}
        
        try:
            context_mapping = {}
            
            for task, context in zip(tasks, contexts):
                dsu = context['dsu']
                if not 'root' in context:
                    context['root'] = dsu.node_store[dsu.create_node()]
                if not 'entropies' in context:
                    context['entropies'] = []
                if not 'varentropies' in context:
                    context['varentropies'] = []
                
                if context['prior_tree'] is not None:
                    context['prior_tree'] = context['prior_tree'].to(self.model.device)

                context_mapping[task.task_id] = context
                                
            for task in tasks:
                task.attempts = task.attempts + 1
            
            task_results = self.fleet_iteration(
                tasks, 0, [context_mapping[t.task_id] for t in tasks], batch_size, verbose, return_trajectory,
                evaluation_mode=evaluation_mode, reward_model=reward_model
            )
            
            for task, results in zip(tasks, task_results):
                entropies, varentropies = [], []
                for b, r in enumerate(results):
                    if isinstance(r, str):
                      print(f"Worker error: {r}")
                      continue # Error on the worker's side
                    
                    stat = r['stats']
                    solution = r['completion']

                    worker_stats = task.stats
                    
                    worker_stats.add(stat)
                    worker_attempt = worker_attempt = batch_size * (task.attempts - 1) + b
                    peak_mem[worker_attempt].append(r['peak_mem'])
                    exec_time[worker_attempt].append(r['exec_time'])
                    
                    if 'trajectory' in r and stat['score'] == 1.0 and worker_attempt != 0:
                        trajectory = r['trajectory']
                        residual_collections[task.task_id].trajectories.append(trajectory)

                    entropy_data = r['entropy_data']
                    worker_entropies, worker_varentropies = entropy_data

                    worker_entropies = worker_entropies[-(10000 // batch_size):]
                    entropies.extend(worker_entropies)

                    worker_varentropies = worker_varentropies[-(10000 //  batch_size):]
                    varentropies.extend(worker_varentropies)

                context = context_mapping[task.task_id]
                
                entropy_threshold, varentropy_threshold = context['threshold']
                target_percentage = min(np.pow(task.attempts, 1/3), 100)

                entropies = sorted(entropies, reverse=True)
                entropy_target_hits = max(0, int(max(len(entropies), 1000) * target_percentage / 100)-1)
                new_entropy_threshold = min(entropies[min(entropy_target_hits, max(len(entropies)-1, 0))], entropy_threshold)

                varentropies = sorted(varentropies, reverse=True)
                varentropy_target_hits = max(0, int(max(len(varentropies), 1000) * target_percentage / 100)-1)
                new_varentropy_threshold = min(varentropies[min(varentropy_target_hits, max(len(varentropies)-1, 0))], varentropy_threshold)

                context['threshold'] = (new_entropy_threshold, new_varentropy_threshold)                        
        except Exception as e:
            print(f"Error while running benchmark: {e}")
            print(f"Traceback:")
            traceback.print_exc()

        gc.collect()
        torch.cuda.empty_cache()
        
        return peak_mem, exec_time, tasks, [context_mapping[t.task_id] for t in tasks], [residual_collections[t.task_id] for t in tasks]
    
    def fleet_iteration(
        self, tasks, rank, contexts, iterations, verbose, return_trajectory, 
        evaluation_mode="ground_truth", reward_model=None
    ):
        results = [[] for t in tasks]
        logit_dim = self.model.config.vocab_size

        workers = []
        for context in contexts:
            worker = FleetWorker(
                rank, context['dsu'], context['root'], context['layer'], context['threshold'], logit_dim,
                prior_tree=context['prior_tree'], strategy='naive', resample_temperature=context['temperature'],
                verbose=verbose, use_reward_penalty=False, return_trajectory=return_trajectory
            )
            workers.append(worker)
    
        for i in range(iterations):
            try:
                torch.cuda.reset_peak_memory_stats()
                mem_before = torch.cuda.memory_allocated()
                start = time.perf_counter()

                worker_results = entropy_search_based_sampler(
                    self.model, self.tokenizer, workers, self.dtype, [task.conversation for task in tasks], verbose=verbose
                )
                end = time.perf_counter()

                mem_after = torch.cuda.memory_allocated()
                peak_mem = torch.cuda.max_memory_allocated()

                for w, (result, task) in enumerate(zip(worker_results, tasks)):
                    worker = workers[w]
                    
                    solution = result['completion']
                    extended_convo = [message for message in task.conversation]
                    extended_convo.append({'role': 'assistant', 'content': solution})
                    
                    algorithm_active_footprint = mem_after - mem_before
                    algorithm_peak_spike = peak_mem - mem_before
    
                    result['default_mem'] = mem_before
                    result['footprint_mem'] = algorithm_active_footprint
                    result['peak_mem'] = peak_mem
                    result['exec_time'] = end - start
                    
                    try:
                        if task.task_data["source"] == "gsm8k":
                            correct = evaluate_gsm8k(task.task_data, extended_convo, verbose=verbose)
                            stats = {'correct': float(correct), 'score': float(correct)}
                        else:
                            stats = evaluate_task(task.task_data, extended_convo, verbose=verbose)
                            stats = {'correct': stats['correct'], 'score': stats['score']}
                    except Exception as e:
                        print(traceback.format_exc())
                        stats = {'correct': False, 'score': 0.0}
    
                    if evaluation_mode == "ground_truth":
                        reward = stats['score']
                    elif evaluation_mode == "orm":
                        reward = reward_model.get_reward.remote(extended_convo)
                        reward = ray.get(reward)
                    elif evaluation_mode == "mixed":
                        reward = reward_model.get_reward.remote(extended_convo)
                        verifier_reward = stats['score']
                        orm_reward = ray.get(reward)
                        smoothed_max = max(orm_reward, verifier_reward) - 0.6 * max(0, (orm_reward - verifier_reward)**2)
                        reward = smoothed_max if verifier_reward != 1.0 else 1.0
                    elif evaluation_mode == "diversity_sampling":
                        reward = 0.0
                    else:
                        raise ValueError("Unknown evaluation mode")
    
                    stats['reward'] = reward
                    stats['completion'] = solution
                    result['stats'] = stats
                    
                    if evaluation_mode == "diversity_sampling":
                        trajectory = worker.finish_iteration(0.0)
                    else:
                        trajectory = worker.finish_iteration(reward)
                        
                    if trajectory is not None:
                        result['trajectory'] = trajectory
        
                    result['entropy_data'] = (copy.copy(worker.entropies), copy.copy(worker.varentropies))
                    results[w].append(result)
            except Exception:
                for w, t in enumerate(tasks):
                    results[w].append(traceback.format_exc())

        del workers
        gc.collect()
        torch.cuda.empty_cache()
        
        return results

    def sampling_job(self, 
        sampling_task_producer, output_queue, 
        temperature, batch_size, max_attempts, verbose, return_trajectory, evaluation_mode, reward_model
    ):
        while True:
            batch = []
            while len(batch) < 4:
                task = ray.get(sampling_task_producer.get.remote())
                if task is None:
                    break

                batch.append(task)

            if len(batch) == 0:
                break

            results = self.sampling_task(batch, temperature, batch_size, max_attempts, verbose, return_trajectory, evaluation_mode, reward_model)
            peak_mem, exec_time, tasks, residual_collections = results
            for i, (task, residual_collection) in enumerate(zip(tasks, residual_collections)):
                output_queue.put((peak_mem, exec_time, task, residual_collection))
    
    def sampling_task(
        self, tasks, temperature, batch_size, max_attempts, 
        verbose, return_trajectory, evaluation_mode, reward_model
    ):
        solutions = [[] for t in tasks]
        peak_mem = {i: [] for i in range(max_attempts * batch_size)}
        exec_time = {i: [] for i in range(max_attempts * batch_size)}

        dsus = {t.task_id: VectorDSU() for t in tasks}
        residual_collections = {t.task_id: ResidualCollection(dsu=dsus[t.task_id]) for t in tasks}
        
        try:
            for attempt in range(max_attempts):
                for task in tasks:
                    task.attempts = attempt + 1

                task_results = self.baseline_iteration(
                    tasks, 0, {}, batch_size, verbose, return_trajectory, temperature=temperature,
                    evaluation_mode=evaluation_mode, reward_model=reward_model
                )

                for task, results in zip(tasks, task_results):
                    for b, r in enumerate(results):
                        if isinstance(r, str):
                          print(f"Worker error: {r}")
                          continue # Error on the worker's side
                        
                        stat = r['stats']
                        solution = r['completion']

                        worker_stats = task.stats
                        worker_stats.add(stat)
                        
                        worker_attempt = batch_size * attempt + b
                        peak_mem[worker_attempt].append(r['peak_mem'])
                        exec_time[worker_attempt].append(r['exec_time'])

                gc.collect()
        except Exception as e:
            print(f"Error while running benchmark: {e}")
            print(f"Traceback:")
            traceback.print_exc()

            gc.collect()
            torch.cuda.empty_cache()

        return peak_mem, exec_time, tasks, [residual_collections[t.task_id] for t in tasks]
    
    def baseline_iteration(
        self, tasks, rank, context, iterations, verbose, return_trajectory, temperature,
        evaluation_mode="ground_truth", reward_model=None
    ):
        results = [[] for t in tasks]

        for i in range(iterations):
            try:
                torch.cuda.reset_peak_memory_stats()
                mem_before = torch.cuda.memory_allocated()
                start = time.perf_counter()
                
                worker_results = temperature_based_sampler(
                    self.model, self.tokenizer, self.dtype, 
                    [task.conversation for task in tasks], 
                    temperature=temperature, verbose=verbose
                )
                end = time.perf_counter()

                mem_after = torch.cuda.memory_allocated()
                peak_mem = torch.cuda.max_memory_allocated()
                
                algorithm_active_footprint = mem_after - mem_before
                algorithm_peak_spike = peak_mem - mem_before

                for w, (result, task) in enumerate(zip(worker_results, tasks)):
                    solution = result['completion']
                    extended_convo = [message for message in task.conversation]
                    extended_convo.append({'role': 'assistant', 'content': solution})
                
                    result['default_mem'] = mem_before
                    result['footprint_mem'] = algorithm_active_footprint
                    result['peak_mem'] = peak_mem
                    result['exec_time'] = end - start
                    
                    try:
                        if task.task_data["source"] == "gsm8k":
                            correct = evaluate_gsm8k(task.task_data, extended_convo, verbose=verbose)
                            stats = {'correct': float(correct), 'score': float(correct)}
                        else:
                            stats = evaluate_task(task.task_data, extended_convo, verbose=verbose)
                            stats = {'correct': stats['correct'], 'score': stats['score']}
                    except Exception as e:
                        print(traceback.format_exc())
                        stats = {'correct': False, 'score': 0.0}
    
                    if evaluation_mode == "ground_truth":
                        reward = stats['score']
                    elif evaluation_mode in ["orm", "diversity_sampling"]:
                        reward = reward_model.get_reward.remote(extended_convo)
                        reward = ray.get(reward)
                    else:
                        raise ValueError("Unknown evaluation mode")
    
                    stats['reward'] = reward
                    stats['completion'] = solution
                    result['stats'] = stats
                    
                    results[w].append(result)
            except Exception:
                for w, t in enumerate(tasks):
                    results[w].append(traceback.format_exc())

        gc.collect()
        torch.cuda.empty_cache()
        
        return results

programmer_prompt_content = """
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

Format:
- [Standalone] Make sure that your answer consists of only one Python function at the top level. Do not wrap with a class or split into multiple functions.
"""

# Some models may need: Use fenced code block syntax from markdown ```python ``` to wrap your code.

programmer_system_prompt = {"role": "system", "content": programmer_prompt_content}

math_prompt_content = """
You are a helpful assistant. Your task is to help solving simple math problems. Try to break the problem into substeps, so it is transparent how
you have arrived to the final solution, just like in example QA pairs.

Format:
- Final answer should be a number, not an expression and is always the final line of the solution, preceded by ####.
"""

math_system_prompt = {"role": "system", "content": math_prompt_content}

def extract_conversation(task):
    conversation = []
        
    if task['source'] in ["livecodebench", "humaneval", "gsm8k_coding"]:
        conversation.append(programmer_system_prompt)
    if task['source'] in ["gsm8k"]:
        conversation.append(math_system_prompt)

    content = ""
    if content == "":
        content = task.get('prompt', "").strip()

    if content == "":
        content = task.get('question_content', "").strip()

    conversation.append({'role': 'user', 'content': content})

    return conversation

@ray.remote
class FleetTaskProvider(object):
    def __init__(self, tasks, context, prior_tree = None):
        self.tasks = tasks
        self.offset = 0
        self.context = context
        self.prior_tree = prior_tree
    
    def set_prior_tree(self, prior_tree):
        self.prior_tree = prior_tree
    
    def get(self):
        if self.offset >= len(self.tasks):
            return None

        task = self.tasks[self.offset]
        conversation = extract_conversation(task)
        task_state = TaskState(task_id=task['task_id'], task_data=task, conversation=conversation)
        task_state.stats = Stats()
        context = self.context
        context['prior_tree'] = self.prior_tree

        self.offset += 1
        return task_state, context

@ray.remote
class SamplingTaskProvider(object):
    def __init__(self, tasks):
        self.tasks = tasks
        self.offset = 0

    def get(self):
        if self.offset >= len(self.tasks):
            return None

        task = self.tasks[self.offset]
        conversation = extract_conversation(task)
        task_state = TaskState(task_id=task['task_id'], task_data=task, conversation=conversation)
        task_state.stats = Stats()
        
        self.offset += 1
        return task_state

In [ ]:
from fleet_ray_worker import FleetActor, RewardActor, FleetTaskProvider, SamplingTaskProvider

In [ ]:
workers = []
dtype = torch.float16
use_reward_model = STRATEGY in ["orm", "mixed"]

budget = device_count

if use_reward_model:
    reward_model = RewardActor.remote()
    budget -= 1
else:
    reward_model = None

for i in range(budget):
    worker = FleetActor.remote(MODEL_NAME, dtype)
    workers.append(worker)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
generation_kwargs = dict(
    max_new_tokens=1024,
    do_sample=False,          
    pad_token_id=tokenizer.pad_token_id,
)

generation_config = GenerationConfig(
    max_new_tokens=1024,
    do_sample=False,          
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
import math
from collections import deque
from pydantic import BaseModel, Field, ConfigDict, field_serializer, field_validator
from typing import List, Dict, Set, Any, Optional

In [ ]:
# Code used to infer optimal temperature for temperature sampling baseline

"""
from skopt import Optimizer

opt = Optimizer([(0.1, 2.0)])
max_attempts = 16
metadata = {
    "model": MODEL_NAME,
    "attempts": max_attempts,
    **generation_config.__dict__
}

tasks = random.sample(lcb_coding_tasks, 20)
xs = []
ys = []

for i in range(10):
    temp = opt.ask()[0]
    xs.append(temp)

    statistics, results, _, _, _ = await benchmark(
        tasks,
        workers, 
        metadata, 
        max_attempts=max_attempts, 
        max_tasks=None, 
        verbose=False,
        batch_size=1,
        group_size=4,
        temperature=temp
    )

    pass_k_total = 0.0
    
    for t in results.items:
        pass_k_total += t.stats.pass_k(max_attempts // 2)
    
    score = pass_k_total / len(results.items) # statistics[0]['accuracy']
    ys.append(score)

    print(f"Temperature: {temp}, Pass@K: {score}")
    opt.tell([temp], -score)

xs = np.array(xs).flatten()
ys = np.array(ys).flatten()

sns.lineplot(x=xs, y=ys)
plt.show()
"""

In [ ]:
def update_pbar_stats(pbar, finished_states):
    correct_count = sum(1 for s in finished_states if s.stats.best_stats['correct'])
    total = len(finished_states)
    if total == 0: return

    scores = [s.stats.best_stats['score'] for s in finished_states]

    pbar.set_postfix({
        'acc': f"{correct_count / total:.2f}",
        'avg_score': f"{sum(scores) / total:.2f}"
    })

def compile_statistics(finished_states):
    correct = sum(1 for s in finished_states if s.stats.best_stats['correct'])
    total = len(finished_states)
    scores = [s.stats.best_stats['score'] for s in finished_states]

    return {
        'accuracy': correct / total if total else 0,
        'avg score': sum(scores) / total if total else 0
    }, [s.conversation for s in finished_states]

In [ ]:
import itertools
import traceback
from ray.util.queue import Queue

async def benchmark(
    tasks_source,
    workers,
    metadata,
    context,
    sampling,
    strategy="ground_truth",
    verbose=False,
    return_trajectory=True,
    use_prior_tree=False,
    max_tasks=None,
    max_attempts=2,
    batch_size=8,
    group_size=4,
):
    if max_tasks is not None and max_tasks < len(tasks_source):
        tasks_source = random.sample(tasks_source, max_tasks)

    progress_bar = tqdm(total=len(tasks_source))
    peak_mem = {i: [] for i in range(max_attempts * batch_size)}
    exec_time = {i: [] for i in range(max_attempts * batch_size)}
    finished_states = []
    residuals = []

    traces = []
    if sampling == "fleet":
        task_provider = FleetTaskProvider.remote(tasks_source, context)
    else:
        task_provider = SamplingTaskProvider.remote(tasks_source)

    queue = Queue()

    for w in workers:
        if sampling == "fleet":
            w.fleet_job.remote(
                task_provider, queue, batch_size, max_attempts, verbose, return_trajectory, strategy, reward_model
            )
        else:
            w.sampling_job.remote(
                task_provider, queue, context.get('temperature', 1.0), batch_size, max_attempts, verbose, return_trajectory, strategy, reward_model
            )

    for i in range(len(tasks_source)):
        try:
            worker_peak_mem, worker_exec_time, task, residual_collection = await queue.get_async()
            finished_states.append(task)

            for attempt, values in worker_peak_mem.items():
                peak_mem[attempt].extend(values)
            for attempt, values in worker_exec_time.items():
                exec_time[attempt].extend(values)

            if len(residual_collection.trajectories) > 0:
                residual_collection.dsu = residual_collection.dsu.to('cpu')
                residuals.append(residual_collection)
            traces.append(residual_collection.dsu)

            progress_bar.update(1)
            update_pbar_stats(progress_bar, finished_states)
        except:
            for w in workers:
                ray.kill(w)
            ray.kill(task_provider)
            print(traceback.format_exc())
            break

    print("Benchmark finished successfully")
    print("Dataset queue finished successfully")

    progress_bar.close()
    statistics = compile_statistics(finished_states)
    results = BenchmarkResults(items=finished_states, metadata=metadata)
    return statistics, results, residuals, peak_mem, exec_time

In [ ]:
residuals_path = None

In [ ]:
from pydantic import TypeAdapter
adapter = TypeAdapter(list[ResidualCollection])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file_path_full = f"Llama-3.2-3B-Instruct.json"

In [ ]:
metadata = {
    "model": MODEL_NAME,
    "attempts": MAX_ATTEMPTS,
    **generation_config.__dict__
}

if residuals_path is None:
    statistics, results, residuals, peak_mem, exec_time = await benchmark(
        math_tasks[1270:],
        workers, 
        metadata,
        context=HYPERPARAMETERS,
        sampling=SAMPLING,
        strategy=STRATEGY,
        max_attempts=MAX_ATTEMPTS,
        max_tasks=None, 
        verbose=False,
        return_trajectory=True,
        batch_size=1,
    )

    results_dict = results.model_dump()

    with open(file_path_full, 'w') as f:
        json.dump(results_dict, f, indent=4)

    print(statistics[0]['accuracy'], statistics[0]['avg score'])

    residuals_path = f"Llama-3.2-3B-Instruct-residuals.json"
    with open(residuals_path, 'w') as f:
        json_data = adapter.dump_json(residuals, indent=4)
        f.write(json_data.decode("utf-8"))

In [ ]:
with open(residuals_path, 'r') as file:
    json_data = file.read()
    residuals = adapter.validate_json(json_data)

In [ ]:
len(residuals)

In [ ]:
from pydantic import BaseModel, ValidationError

with open(file_path_full, 'r') as f:
    json_data = json.load(f)

    for task in json_data['items']:
        try:
            results = BenchmarkResults(**json_data)
        except ValidationError as e:
            print(e.errors())

            for error in e.errors():
                print(f"Field: {error['loc']}, Error: {error['msg']}")

In [ ]:
xs = list(range(1, MAX_ATTEMPTS+1))
ys = []

for k in xs:
    pass_k_total = 0.0
    for t in results.items:
        pass_k_total += t.stats.pass_k(k)

    ys.append(pass_k_total / len(results.items))

sns.lineplot(x=xs, y=ys)
plt.show()

xs = list(range(1, MAX_ATTEMPTS+1))
ys = []

for k in xs:
    pass_k_total = 0.0
    for t in results.items:
        pass_k_total += t.stats.pass_k_noisy(k)

    ys.append(pass_k_total / len(results.items))

sns.lineplot(x=xs, y=ys)
plt.show()

xs = list(range(1, MAX_ATTEMPTS+1))
ys = []

for k in xs:
    pass_k_total = 0.0
    for t in results.items:
        pass_k_total += t.stats.pass_k_orm(k)

    ys.append(pass_k_total / len(results.items))

sns.lineplot(x=xs, y=ys)
plt.show()

xs = list(range(1, MAX_ATTEMPTS+1))
ys = []

for k in xs:
    pass_k_total = 0.0
    for t in results.items:
        pass_k_total += t.stats.rm_k(k)

    ys.append(pass_k_total / len(results.items))

sns.lineplot(x=xs, y=ys)
plt.show()

In [ ]:
sns.violinplot(peak_mem)
plt.show()

sns.violinplot(exec_time)
plt.show()

In [ ]:
from statistics import median 

print(statistics[0], median(itertools.chain(*peak_mem.values())), median(itertools.chain(*exec_time.values())))
print(statistics[0], max(itertools.chain(*peak_mem.values())), max(itertools.chain(*exec_time.values())))

In [ ]:
with open(file_path_full, 'w') as f:
    json.dump(results_dict, f, indent=4)

In [ ]:
stats_only = {'items': [{'id': t.task_id, 'stats': t.stats.model_dump()} for t in results.items]}
with open("Test" + file_path_full, 'w') as f:
    json.dump(stats_only, f, indent=4)